## 암묵적인 데이터에 대한 행렬 분해(Implicit Matrix Factorization, IMF)

In [1]:
import sys; sys.path.insert(0, '..')

from test_util.data_loader import DataLoader
from test_util.metric_calculator import MetricCalculator

In [2]:
data_loader = DataLoader(num_users=1000, num_test_items=5, data_path='../data/ml-10m/')
movielens = data_loader.load()

In [3]:
from test_src.imf import IMFRecommender
recommender = IMFRecommender()
recommend_result = recommender.recommend(movielens)

c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.00299072265625 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 65.96it/s, loss=0.00963]


Item factors shape: (997, 10)
User factors shape: (4983, 10)
Input matrix shape: (4983, 997)


In [4]:
metric_calculator = MetricCalculator()
metrics = metric_calculator.calc(
    movielens.test.rating.tolist(), recommend_result.rating.tolist(),
    movielens.test_user2items, recommend_result.user2items, k=10
)
print(metrics)

rmse=0.000, Precision@K=0.002, Recall@K=0.006


In [5]:
# 평가 수의 임계값과 정밀도 관계
for minimum_num_rating in [0, 10]:
    recommender = IMFRecommender()
    recommend_result = recommender.recommend(movielens, minimum_num_rating=minimum_num_rating)
    metrics = metric_calculator.calc(
        movielens.test.rating.tolist(), recommend_result.rating.tolist(),
        movielens.test_user2items, recommend_result.user2items, k=10
    )
    print(metrics)

c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.001085042953491211 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 73.58it/s, loss=0.00963]


Item factors shape: (997, 10)
User factors shape: (4983, 10)
Input matrix shape: (4983, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.006


c:\eunjoo\Projects\book_rec_sys_2024\test_notebook\..\test_src\imf.py:34: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  movielens_train_high_rating = filtered_movielens_train[dataset.train.rating >= 4]
c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.0009970664978027344 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 126.68it/s, loss=0.0186]

Item factors shape: (997, 10)
User factors shape: (2319, 10)
Input matrix shape: (2319, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.005


In [6]:
for minimum_num_rating in [0, 10]:
    # 데이터 필터링 및 학습/추천
    recommend_result = recommender.recommend(movielens, minimum_num_rating=minimum_num_rating)
    
    # 학습 데이터와 평가 데이터 간 동기화 확인
    filtered_users = set(recommend_result.user2items.keys())
    filtered_items = set(item for items in recommend_result.user2items.values() for item in items)
    
    test_users = set(movielens.test_user2items.keys())
    test_items = set(item for items in movielens.test_user2items.values() for item in items)
    
    # 교집합을 기반으로 평가 진행
    valid_test_users = filtered_users.intersection(test_users)
    valid_test_items = filtered_items.intersection(test_items)
    
    # 평가
    filtered_test_user2items = {u: movielens.test_user2items[u] for u in valid_test_users}
    metrics = metric_calculator.calc(
        movielens.test.rating.tolist(), recommend_result.rating.tolist(),
        filtered_test_user2items, recommend_result.user2items, k=10
    )
    print(metrics)

c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.001993417739868164 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 60.06it/s, loss=0.00963]


Item factors shape: (997, 10)
User factors shape: (4983, 10)
Input matrix shape: (4983, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.006


c:\eunjoo\Projects\book_rec_sys_2024\test_notebook\..\test_src\imf.py:34: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  movielens_train_high_rating = filtered_movielens_train[dataset.train.rating >= 4]
c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.0019941329956054688 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 110.55it/s, loss=0.0186]

Item factors shape: (997, 10)
User factors shape: (2319, 10)
Input matrix shape: (2319, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.005


In [7]:
# alpha와 정밀도의 관계
for alpha in [0.5, 1.0, 2.0, 5.0]:
    recommend_result = recommender.recommend(movielens, alpha=alpha)
    metrics = metric_calculator.calc(
    movielens.test.rating.tolist(), recommend_result.rating.tolist(),
    movielens.test_user2items, recommend_result.user2items, k=10)
    print(metrics)

c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.0020034313201904297 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 70.16it/s, loss=0.00542]


Item factors shape: (997, 10)
User factors shape: (4983, 10)
Input matrix shape: (4983, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.005


c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.0023276805877685547 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 74.73it/s, loss=0.00963]


Item factors shape: (997, 10)
User factors shape: (4983, 10)
Input matrix shape: (4983, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.006


c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.001993417739868164 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 65.28it/s, loss=0.0162]


Item factors shape: (997, 10)
User factors shape: (4983, 10)
Input matrix shape: (4983, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.006


c:\Users\eunjo\anaconda3\envs\rec_sys_2024\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed lil_matrix instead. Converting to CSR took 0.002172231674194336 seconds
  warnings.warn(
100%|██████████| 50/50 [00:00<00:00, 63.71it/s, loss=0.0293]

Item factors shape: (997, 10)
User factors shape: (4983, 10)
Input matrix shape: (4983, 997)
rmse=0.000, Precision@K=0.002, Recall@K=0.007
